In [1]:
import numpy as np
import time

# ── Parámetros ────────────────────────────────────────────
N = 10
NPT = 600
H = 0.050
ALFA = 0.2
ANCHO = 1.0

# ── Variables ─────────────────────────────────────────────
# Usamos tamaño N+1 para mantener los índices de 1 a N idénticos a Fortran
Y = np.zeros(N + 1)
Y_NEW = np.zeros(N + 1)
A = np.zeros(N + 1)
B = np.zeros(N + 1)
C = np.zeros(N + 1)
D = np.zeros(N + 1)   # sistema tridiagonal

DX = ANCHO / N
T = 0.0
R = (ALFA**2) * H / (2.0 * DX**2)   # número de Fourier/2

# ── Cronómetro ────────────────────────────────────────────
t_inicio = time.perf_counter()

# Abrir archivo (el bloque 'with' lo cierra automáticamente al terminar)
with open('CNPY.dat', 'w') as f:
    
    # ── Condición inicial ──────────────────────────────────────
    for i in range(2, N + 1):
        Y[i] = 100.0
    Y[1] = 0.0   # frontera fría fija

    # ── Escribir t=0 ──────────────────────────────────────────
    FR_DER = (4.0 * Y[N] - Y[N - 1]) / 3.0
    
    # Replicando el formato Fortran (15F10.4)
    linea_t0 = f"{T:10.4f}" + "".join([f"{Y[i]:10.4f}" for i in range(1, N + 1)]) + f"{FR_DER:10.4f}\n"
    f.write(linea_t0)

    # ── Ciclo temporal ────────────────────────────────────────
    for j in range(1, NPT + 1):
        
        # Construir sistema tridiagonal A·Y_NEW = D
        # Frontera x=0: Dirichlet T1=0 siempre
        A[1] = 0.0
        B[1] = 1.0
        C[1] = 0.0
        D[1] = 0.0

        # Puntos interiores i=2..N-1
        for i in range(2, N):
            A[i] = -R
            B[i] =  1.0 + 2.0 * R
            C[i] = -R
            D[i] =  R * Y[i - 1] + (1.0 - 2.0 * R) * Y[i] + R * Y[i + 1]

        # Frontera x=l: Neumann dT/dx=0
        A[N] = -R / 3.0
        B[N] =  1.0 + (2.0 * R) / 3.0
        C[N] =  0.0
        D[N] =  (R / 3.0) * Y[N - 1] + (1.0 - (2.0 * R) / 3.0) * Y[N]

        # ── Eliminación de Thomas (TDMA) ──────────────────────
        for i in range(2, N + 1):
            m = A[i] / B[i - 1]
            B[i] = B[i] - m * C[i - 1]
            D[i] = D[i] - m * D[i - 1]

        # Sustitución regresiva
        Y_NEW[N] = D[N] / B[N]
        # Bucle hacia atrás: desde N-1 hasta 1
        for i in range(N - 1, 0, -1):
            Y_NEW[i] = (D[i] - C[i] * Y_NEW[i + 1]) / B[i]

        # Actualización temporal
        T = T + H
        for i in range(1, N + 1):
            Y[i] = Y_NEW[i]

        FR_DER = (4.0 * Y[N] - Y[N - 1]) / 3.0
        
        # Escribir paso temporal
        linea = f"{T:10.4f}" + "".join([f"{Y[i]:10.4f}" for i in range(1, N + 1)]) + f"{FR_DER:10.4f}\n"
        f.write(linea)

# ── Fin del Cronómetro ────────────────────────────────────
t_fin = time.perf_counter()
t_ejecucion = t_fin - t_inicio

# ── Prints de salida editados ─────────────────────────────
print("=================================")
print("  MELIN — Crank-Nicolson Python")
print("---------------------------------")
print(f"  R (Fourier/2) = {R:.4f}")
print("  Estabilidad   : INCONDICIONAL")
print(f"  Tiempo        : {t_ejecucion:.6f} s")
print("  Archivo       : CNPY.dat")
print("=================================")

  MELIN — Crank-Nicolson Python
---------------------------------
  R (Fourier/2) = 0.1000
  Estabilidad   : INCONDICIONAL
  Tiempo        : 0.014361 s
  Archivo       : CNPY.dat
